# 🚗 Pipeline Final v3 - VMMRdb

**Détection de véhicules (YOLO) + Classification marque (EfficientNet VMMRdb)**

```
Image → YOLO (détection) → Crops véhicules → EfficientNet (classification) → Marque
```

**Modèles:**
- YOLO fine-tuné sur dataset parking
- EfficientNet-B0 fine-tuné sur VMMRdb (22 marques, 82.68% accuracy)

## 1. Imports et Configuration

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

from torchvision.models import efficientnet_b0
from ultralytics import YOLO

import albumentations as A
from albumentations.pytorch import ToTensorV2

print("✅ Imports OK")

In [ ]:
# === CONFIGURATION ===

# Chemins des modèles
YOLO_MODEL_PATH = "DETECT_MODE_CARS/models/yolo_parking_finetuned/weights/best.pt"
EFFICIENTNET_MODEL_PATH = "models/efficientnet_vmmrdb/best_model_vmmrdb.pth"

# Paramètres YOLO
YOLO_CONFIDENCE = 0.5
YOLO_IOU = 0.45

# Paramètres EfficientNet
IMG_SIZE = 224
NUM_CLASSES = 22

# Classes (ordre du LabelEncoder)
CLASSES = ['audi', 'bmw', 'citroen', 'ford', 'honda', 'hyundai', 'jaguar',
           'kia', 'landrover', 'mazda', 'mercedes benz', 'mini', 'mitsubishi',
           'nissan', 'peugeot', 'porsche', 'renault', 'saab', 'suzuki',
           'toyota', 'volkswagen', 'volvo']

# Dossier de sortie
OUTPUT_DIR = Path("outputs/pipeline_results_v3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"📊 Classes: {NUM_CLASSES} marques")

## 2. Chargement YOLO

In [ ]:
print(f"📦 Chargement YOLO: {YOLO_MODEL_PATH}")
yolo_model = YOLO(YOLO_MODEL_PATH)
print("✅ YOLO chargé")

## 3. Chargement EfficientNet VMMRdb

In [ ]:
print(f"📦 Chargement EfficientNet: {EFFICIENTNET_MODEL_PATH}")

# Charger le checkpoint
checkpoint = torch.load(EFFICIENTNET_MODEL_PATH, map_location=device)

# Créer le modèle
efficientnet_model = efficientnet_b0(weights=None)
efficientnet_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(1280, NUM_CLASSES)
)

# Charger les poids
efficientnet_model.load_state_dict(checkpoint['model_state_dict'])
efficientnet_model = efficientnet_model.to(device)
efficientnet_model.eval()

print(f"✅ EfficientNet chargé")
print(f"   Val Accuracy: {checkpoint['val_acc']:.2f}%")
print(f"   Classes: {checkpoint.get('label_encoder_classes', CLASSES)}")

## 4. Transformations

In [ ]:
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

print("✅ Transformations définies")

## 5. Fonctions du Pipeline

In [ ]:
def detect_vehicles(image):
    """
    Détecte les véhicules avec YOLO
    """
    results = yolo_model(
        image,
        conf=YOLO_CONFIDENCE,
        iou=YOLO_IOU,
        verbose=False
    )
    
    detections = []
    
    for result in results:
        boxes = result.boxes
        if boxes is None:
            continue
        
        for i in range(len(boxes)):
            bbox = boxes.xyxy[i].cpu().numpy().astype(int)
            conf = float(boxes.conf[i].cpu().numpy())
            cls_id = int(boxes.cls[i].cpu().numpy())
            cls_name = result.names[cls_id]
            
            detections.append({
                'bbox': tuple(bbox),
                'confidence': conf,
                'class_id': cls_id,
                'class_name': cls_name
            })
    
    return detections

print("✅ Fonction detect_vehicles")

In [ ]:
def classify_vehicle(crop):
    """
    Classifie la marque du véhicule avec EfficientNet
    """
    # BGR → RGB
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    
    # Transformer
    transformed = transform(image=crop_rgb)
    tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Inférence
    with torch.no_grad():
        output = efficientnet_model(tensor)
        probs = torch.softmax(output, dim=1)
        top5_probs, top5_indices = torch.topk(probs, k=min(5, len(CLASSES)))
    
    # Décoder
    top5 = []
    for i in range(len(top5_indices[0])):
        idx = top5_indices[0][i].item()
        prob = top5_probs[0][i].item()
        top5.append((CLASSES[idx], prob))
    
    return {
        'make': top5[0][0],
        'confidence': top5[0][1],
        'top5': top5
    }

print("✅ Fonction classify_vehicle")

In [ ]:
def create_visualization(image, results):
    """
    Crée l'image annotée
    """
    vis = image.copy()
    
    for i, res in enumerate(results):
        x1, y1, x2, y2 = res['detection']['bbox']
        conf = res['classification']['confidence']
        make = res['classification']['make'].upper()
        
        # Couleur selon confiance
        if conf > 0.7:
            color = (0, 255, 0)    # Vert
        elif conf > 0.4:
            color = (0, 165, 255)  # Orange
        else:
            color = (0, 0, 255)    # Rouge
        
        # Rectangle
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 3)
        
        # Label
        label = f"{make} ({conf:.0%})"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
        cv2.rectangle(vis, (x1, y1 - th - 15), (x1 + tw + 10, y1), color, -1)
        cv2.putText(vis, label, (x1 + 5, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # Titre
    title = f"Detection: {len(results)} vehicule(s)"
    cv2.putText(vis, title, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    
    return vis

print("✅ Fonction create_visualization")

In [ ]:
def save_json(results, image_path, output_path):
    """
    Sauvegarde les résultats en JSON
    """
    data = {
        'image_path': str(image_path),
        'timestamp': datetime.now().isoformat(),
        'num_vehicles': len(results),
        'vehicles': []
    }
    
    for res in results:
        vehicle = {
            'detection': {
                'bbox': [int(x) for x in res['detection']['bbox']],
                'confidence': float(res['detection']['confidence']),
                'class': str(res['detection']['class_name'])
            },
            'classification': {
                'make': res['classification']['make'],
                'confidence': float(res['classification']['confidence']),
                'top5': [
                    {'make': str(m), 'confidence': float(c)}
                    for m, c in res['classification']['top5']
                ]
            }
        }
        data['vehicles'].append(vehicle)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("✅ Fonction save_json")

In [ ]:
def process_image(image_path, save_output=True):
    """
    Pipeline complet: détection + classification
    """
    import time
    start = time.time()
    
    # Charger l'image
    image_path = Path(image_path)
    image = cv2.imread(str(image_path))
    
    if image is None:
        raise ValueError(f"Impossible de lire: {image_path}")
    
    print(f"\n🔍 Traitement: {image_path.name}")
    
    # 1. Détection
    detections = detect_vehicles(image)
    print(f"   📦 {len(detections)} véhicule(s) détecté(s)")
    
    # 2. Classification
    results = []
    
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['bbox']
        crop = image[y1:y2, x1:x2]
        
        if crop.size == 0:
            continue
        
        classification = classify_vehicle(crop)
        
        results.append({
            'detection': det,
            'classification': classification
        })
        
        print(f"   🚗 Véhicule {i+1}: {classification['make'].upper()} ({classification['confidence']:.1%})")
    
    # 3. Visualisation
    vis_image = create_visualization(image, results)
    
    # 4. Sauvegarder
    if save_output:
        output_img = OUTPUT_DIR / f"{image_path.stem}_result.jpg"
        cv2.imwrite(str(output_img), vis_image)
        
        output_json = OUTPUT_DIR / f"{image_path.stem}_result.json"
        save_json(results, image_path, output_json)
        
        print(f"   💾 Sauvegardé: {output_img.name}")
    
    elapsed = (time.time() - start) * 1000
    print(f"   ⏱️ Temps: {elapsed:.1f}ms")
    
    return {
        'image_path': str(image_path),
        'num_vehicles': len(results),
        'vehicles': results,
        'visualization': vis_image,
        'processing_time_ms': elapsed
    }

print("✅ Fonction process_image")

In [ ]:
def show_result(result):
    """
    Affiche le résultat
    """
    img_rgb = cv2.cvtColor(result['visualization'], cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(14, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Résultat: {result['num_vehicles']} véhicule(s)")
    plt.show()
    
    # Résumé
    print("\n" + "="*50)
    print("📊 RÉSUMÉ")
    print("="*50)
    
    for i, v in enumerate(result['vehicles']):
        print(f"\n🚗 Véhicule {i+1}:")
        print(f"   Marque: {v['classification']['make'].upper()}")
        print(f"   Confiance: {v['classification']['confidence']:.1%}")
        print(f"   Top 3:")
        for make, prob in v['classification']['top5'][:3]:
            print(f"      - {make}: {prob:.1%}")

print("✅ Fonction show_result")

## 6. Test du Pipeline

**Modifie le chemin de l'image ci-dessous:**

In [ ]:
# === CHEMIN DE L'IMAGE À TESTER ===
IMAGE_PATH = "chemin/vers/ton/image.jpg"  # <-- MODIFIE ICI

# Lancer le pipeline
result = process_image(IMAGE_PATH)

# Afficher
show_result(result)

## 7. Test sur le dataset de validation (optionnel)

In [ ]:
# Tester sur quelques images du dataset parking
from pathlib import Path
import random

# Images de test
test_dir = Path("DETECT_MODE_CARS/Data/parking_vehicles/valid/images")

if test_dir.exists():
    images = list(test_dir.glob("*.jpg"))[:5]
    
    for img_path in images:
        result = process_image(img_path)
        show_result(result)
else:
    print(f"⚠️ Dossier non trouvé: {test_dir}")

---

## 📋 Résumé du Pipeline

| Composant | Modèle | Performance |
|-----------|--------|-------------|
| Détection | YOLO fine-tuné | Excellente |
| Classification | EfficientNet VMMRdb | 82.68% accuracy |

**22 marques supportées:**
```
Audi, BMW, Citroën, Ford, Honda, Hyundai, Jaguar, Kia, Land Rover,
Mazda, Mercedes-Benz, Mini, Mitsubishi, Nissan, Peugeot, Porsche,
Renault, Saab, Suzuki, Toyota, Volkswagen, Volvo
```

**Outputs:**
- `outputs/pipeline_results_v3/{image}_result.jpg` — Image annotée
- `outputs/pipeline_results_v3/{image}_result.json` — Résultats JSON